# Demo 01 - Runge's Phenomenon & Chebyshev Nodes

**Standards 1-2** (interpolation construction & error) - Weeks 1-2.

**The question.** More data points should mean a better fit... shouldn't it? For
equally spaced nodes on a smooth function, the answer can be an emphatic *no*.
This demo lets you *watch* a high-degree interpolant diverge, then fixes it by
moving the nodes.

In [ ]:
# --- Colab / Jupyter setup ------------------------------------------------
# Enable interactive ipywidgets sliders. The try/except lets this same
# notebook run in plain Jupyter (outside Colab) without error.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Dropdown

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 5)

## The mechanism

For a function $f$ with $n+1$ continuous derivatives, the interpolation error at
$x$ is

$$ f(x) - p_n(x) \;=\; \frac{f^{(n+1)}(\xi)}{(n+1)!}\,\prod_{k=0}^{n}(x - x_k). $$

Two competing factors as $n$ grows:

- the derivative factor $f^{(n+1)}(\xi)/(n+1)!$, and
- the **node polynomial** $\omega(x)=\prod_k (x-x_k)$, whose size depends entirely
  on *where* you put the nodes.

For **equispaced** nodes, $\omega$ develops huge peaks near the endpoints, and for
Runge's function the derivative factor does not shrink fast enough to compensate -
so the error blows up. **Chebyshev** nodes cluster near the endpoints and
minimise the maximum of $|\omega|$, keeping the product tame. That is the whole
story you are about to see on screen.

In [ ]:
# ---------------------------------------------------------------------------
# Interpolation nodes
# ---------------------------------------------------------------------------
# A degree-n polynomial interpolant is pinned down by n+1 data points.
# WHERE we place those points matters enormously. Two natural choices:

def equispaced_nodes(a, b, n):
    """Return n+1 equally spaced nodes on the interval [a, b].

    The 'obvious' choice: chop [a, b] into n equal pieces. It looks innocent
    but is the source of Runge's phenomenon demonstrated below.

    Parameters
    ----------
    a, b : float   endpoints of the interval
    n    : int     polynomial degree (so we get n+1 nodes)
    """
    return np.linspace(a, b, n + 1)


def chebyshev_nodes(a, b, n):
    """Return n+1 Chebyshev nodes (first kind) on [a, b].

    These are x-projections of equally spaced points on a semicircle, so they
    CLUSTER near the endpoints. That clustering is exactly what tames the wild
    endpoint oscillations of equispaced interpolation.

    On [-1, 1]:  x_k = cos((2k+1)/(2(n+1)) * pi),  k = 0..n, then rescaled.
    """
    k = np.arange(n + 1)
    x_std = np.cos((2 * k + 1) / (2 * (n + 1)) * np.pi)   # nodes on [-1, 1]
    return 0.5 * (a + b) + 0.5 * (b - a) * x_std          # rescale to [a, b]

In [ ]:
# ---------------------------------------------------------------------------
# The interpolating polynomial, evaluated via the Lagrange formula
# ---------------------------------------------------------------------------
def lagrange_eval(nodes, values, x):
    """Evaluate the polynomial interpolating (nodes[i], values[i]) at x.

    Lagrange form:  p(x) = sum_i values[i] * L_i(x), where L_i is 1 at node i
    and 0 at every other node:
        L_i(x) = prod_{j != i} (x - x_j) / (x_i - x_j).

    The clearest way to SEE interpolation, if not the fastest. (Newton's
    divided-difference form -- demo 02 -- is the efficient cousin.)

    Parameters
    ----------
    nodes  : array (m,)  distinct x-coordinates of the data
    values : array (m,)  y-coordinates, values[i] = f(nodes[i])
    x      : array       points at which to evaluate the interpolant
    """
    nodes  = np.asarray(nodes,  dtype=float)
    values = np.asarray(values, dtype=float)
    x      = np.asarray(x,      dtype=float)

    result = np.zeros_like(x)
    m = len(nodes)
    for i in range(m):
        Li = np.ones_like(x)                 # build L_i(x) as a running product
        for j in range(m):
            if j != i:
                Li *= (x - nodes[j]) / (nodes[i] - nodes[j])
        result += values[i] * Li
    return result

In [ ]:
# ---------------------------------------------------------------------------
# Test functions to interpolate
# ---------------------------------------------------------------------------
def runge(x):
    """Runge's function 1/(1 + 25 x^2) on [-1, 1] -- the classic troublemaker:
    perfectly smooth, yet equispaced interpolation of it DIVERGES with degree."""
    return 1.0 / (1.0 + 25.0 * x**2)

def sine(x):
    """A benign, well-behaved function for contrast."""
    return np.sin(np.pi * x)

# Registry so the interactive dropdown can pick a function by name.
TEST_FUNCTIONS = {"Runge 1/(1+25x^2)": runge, "sin(pi x)": sine}
NODE_BUILDERS  = {"equispaced": equispaced_nodes, "Chebyshev": chebyshev_nodes}

In [ ]:
# ---------------------------------------------------------------------------
# One figure: the function, its interpolant, the nodes, and the error curve
# ---------------------------------------------------------------------------
def show_interpolation(degree=12, node_type="equispaced",
                       func_name="Runge 1/(1+25x^2)", a=-1.0, b=1.0):
    """Plot f, its degree-`degree` interpolant on the chosen nodes, and |f - p|.

    Also prints the maximum error over a fine grid, so you can watch the number
    grow (equispaced) or shrink (Chebyshev) as you slide the degree.
    """
    f       = TEST_FUNCTIONS[func_name]
    nodefn  = NODE_BUILDERS[node_type]

    nodes   = nodefn(a, b, degree)           # n+1 interpolation nodes
    values  = f(nodes)                       # sample the function there
    xx      = np.linspace(a, b, 1000)        # dense grid for plotting
    p       = lagrange_eval(nodes, values, xx)
    err     = np.abs(f(xx) - p)
    max_err = err.max()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

    ax1.plot(xx, f(xx), "k-", lw=2, label="f(x)")
    ax1.plot(xx, p, "r--", lw=2, label=f"interpolant p (deg {degree})")
    ax1.plot(nodes, values, "bo", ms=6, label="nodes")
    ax1.set_title(f"{func_name} -- {node_type} nodes")
    ax1.legend(loc="upper right"); ax1.set_xlabel("x")

    ax2.semilogy(xx, err + 1e-18, "m-")      # +tiny value keeps log scale happy
    ax2.set_title(f"pointwise error |f - p|   (max = {max_err:.3e})")
    ax2.set_xlabel("x"); ax2.set_ylabel("error (log scale)")

    plt.tight_layout(); plt.show()

# A first static call so the notebook shows something before you touch sliders.
show_interpolation(degree=12, node_type="equispaced")

## Try it

Start with **equispaced** nodes on **Runge** and push the degree past ~15: the red
interpolant grows huge wiggles near $x=\pm 1$ and the error plot climbs. Now flip
the `nodes` dropdown to **Chebyshev** at the same degree - the wiggles vanish and
the error drops by orders of magnitude. For contrast, try $\sin(\pi x)$, which is
so well-behaved that even equispaced nodes converge nicely.

In [ ]:
# ---------------------------------------------------------------------------
# Interactive version -- drag the sliders in Colab
# ---------------------------------------------------------------------------
# Slide `degree` up with equispaced nodes on the Runge function and watch the
# error EXPLODE near the endpoints. Switch to Chebyshev and watch it collapse.
interact(
    show_interpolation,
    degree=IntSlider(min=2, max=40, step=1, value=12, description="degree n"),
    node_type=Dropdown(options=list(NODE_BUILDERS), value="equispaced",
                       description="nodes"),
    func_name=Dropdown(options=list(TEST_FUNCTIONS),
                        value="Runge 1/(1+25x^2)", description="function"),
    a=(-2.0, 0.0, 0.5), b=(0.0, 2.0, 0.5),
);

## Takeaways

- Polynomial interpolation on **equispaced** nodes can *diverge* even for a smooth
  function (Runge's phenomenon) - higher degree is not automatically better.
- The error formula localises the blame on the **node polynomial** $\omega(x)$.
- **Chebyshev** nodes minimise $\max|\omega|$ and restore rapid convergence.
- Lesson for later demos: the *placement* of information is a design choice, not a
  given - the same theme returns in Gaussian quadrature (demo 13).